# 11. Scaled Multi-Hop QA from Text: Vector RAG vs Graph RAG on MuSiQue and 2WikiMultiHopQA

**Presentation map:** Slide 32 (vector RAG is the baseline to beat), Slide 33 (graph construction from text),
Slide 36 (graphs help on multi-hop questions), Slide 38 (decision rule by question type), Slide 41 (multi-hop
traversal).

Notebook 8 showed a large graph advantage on MetaQA, but MetaQA *hands the system a clean, complete knowledge graph*
and asks templated questions — the most graph-friendly setting possible. This notebook removes that advantage: the
graph must be **built from raw Wikipedia paragraphs by the LLM**, so every extraction error counts against it.

**Benchmarks** (the standard HippoRAG / HippoRAG 2 evaluation sets, Hugging Face `osunlp/HippoRAG_2`):

| Benchmark | Questions | Passage pool | What makes it hard |
|---|---|---|---|
| **MuSiQue** (answerable, dev) | 1,000 (2-, 3- and 4-hop) | 11,656 paragraphs | questions composed so that no single paragraph answers them; many distractors |
| **2WikiMultiHopQA** (dev) | 1,000 (compositional, comparison, inference, bridge-comparison) | 6,119 paragraphs | two-entity chains built from Wikidata relations |

**Two retrievers, one answer prompt, same budget of 5 passages:**
- **(a) Vector RAG** — `text-embedding-3-small` over every passage, top-5 by cosine similarity.
- **(b) Graph RAG (HippoRAG-2-style)** — the LLM extracts entities and subject–predicate–object triples from every
  passage (one structured call per passage); entities with the same normalised name are merged, and near-duplicate
  names (embedding similarity ≥ 0.9) get a synonym edge. At query time the question is matched against the triples,
  the entities of the best-matching triples seed a **personalised PageRank** (damping 0.5, plus a small weight on the
  dense passage scores, as in HippoRAG 2), and the 5 highest-ranked passages go to the answerer.

**Scoring:** exact match (EM) and token F1 against the gold answer and its aliases — the standard metrics of both
benchmarks, computed in code (no LLM judge). Retrieval quality is measured as **recall@5**: the share of the gold
supporting paragraphs among the 5 retrieved passages.

**Model:** `gpt-6-luna`, no reasoning, temperature 0, through the OpenAI **Batch API** (helper `smash_batch.py`).
Everything is cached in `data/benchmarks/musique_2wiki/cache/`; the default `SMASH_MODE=replay` re-runs the saved
results in about a minute without calling the API.

In [1]:
import json, os, re, string, time, random
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/musique_2wiki"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
N_Q = 25 if TEST else 1000           # questions per benchmark (test mode: the first 25 of the same fixed order)

def load(name):
    qs = json.load(open(f"{BASE}/{name}.json"))
    corpus = json.load(open(f"{BASE}/{name}_corpus.json"))
    return qs, corpus

DATA = {}
for name in ("musique", "2wikimultihopqa"):
    qs, corpus = load(name)
    for q in qs:                                     # one schema for both benchmarks
        if name == "musique":
            q["qid"], q["golds"] = q["id"], [q["answer"]] + q.get("answer_aliases", [])
            q["support"] = {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"] if p["is_supporting"]}
            q["group"] = q["id"].split("hop")[0] + "-hop"
        else:
            q["qid"], q["golds"] = q["_id"], [q["answer"]]
            ctx = {t: " ".join(s) for t, s in q["context"]}
            q["support"] = {t + "\n" + ctx[t] for t in {t for t, _ in q["supporting_facts"]} if t in ctx}
            q["group"] = q["type"]
    DATA[name] = {"questions": qs[:N_Q], "corpus": corpus}
    print(f"{name}: {len(qs):,} questions, {len(corpus):,} passages | groups:",
          dict(Counter(q["group"] for q in qs)))
print("MODE =", MODE)

musique: 1,000 questions, 11,656 passages | groups: {'2-hop': 518, '3-hop': 316, '4-hop': 166}
2wikimultihopqa: 1,000 questions, 6,119 passages | groups: {'compositional': 413, 'comparison': 244, 'bridge_comparison': 235, 'inference': 108}
MODE = replay


## The passage pools

Each benchmark is indexed as one pool. In test mode the pool is cut down to the passages of the test questions, so
the whole pipeline can be checked for a few cents.

In [2]:
for name, d in DATA.items():
    passages = [p["title"] + "\n" + p["text"] for p in d["corpus"]]
    if TEST:
        keep = set()
        for q in d["questions"]:
            if name == "musique":
                keep |= {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"]}
            else:
                keep |= {t + "\n" + " ".join(s) for t, s in q["context"]}
        passages = [p for p in passages if p in keep]
    d["passages"] = passages
    d["pid"] = {p: i for i, p in enumerate(passages)}
    found = np.mean([len(q["support"] & set(d["pid"])) / max(len(q["support"]), 1) for q in d["questions"]])
    print(f"{name}: {len(passages):,} passages in the pool; gold supporting paragraphs found in the pool: {found:.1%}")

musique: 11,656 passages in the pool; gold supporting paragraphs found in the pool: 100.0%
2wikimultihopqa: 6,119 passages in the pool; gold supporting paragraphs found in the pool: 100.0%


## Index (a): dense embeddings — the vector baseline's whole indexing cost

In [3]:
for name, d in DATA.items():
    d["P"] = llm.embed(f"{name}_passages", d["passages"])
    d["Q"] = llm.embed(f"{name}_questions", [q["question"] for q in d["questions"]])
    print(f"{name}: passage matrix {d['P'].shape}, question matrix {d['Q'].shape}")

musique: passage matrix (11656, 1536), question matrix (1000, 1536)
2wikimultihopqa: passage matrix (6119, 1536), question matrix (1000, 1536)


## Index (b): LLM triple extraction → entity–passage graph (Slide 33)

One structured-output request per passage (OpenIE, as in HippoRAG). All requests of a benchmark go out as one
Batch API job. Every entity keeps the ids of the passages it came from (provenance), so the graph can always point
back to its source text.

In [4]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str

class OpenIE(BaseModel):
    entities: list[str] = Field(description="named entities, dates and numbers mentioned in the passage")
    triples: list[Triple] = Field(description="factual subject-predicate-object triples stated in the passage")

OPENIE_SYS = ("Extract an open knowledge graph from a Wikipedia passage. List the named entities (people, places, "
              "organisations, works, events, dates, numbers) and every factual triple the passage states. Use full, "
              "canonical entity names (resolve pronouns to the entity they refer to); keep dates and numbers exact. At most "
              "40 triples, most important first.")

TRIPLE_RE = re.compile(r'\{\s*"subject":\s*"((?:[^"\\]|\\.)*)",\s*"predicate":\s*"((?:[^"\\]|\\.)*)",\s*"object":\s*"((?:[^"\\]|\\.)*)"\s*\}')

def salvage(text):
    # An output cut off at the token limit still contains complete triples: keep those.
    return {"entities": [], "triples": [{"subject": a, "predicate": b, "object": c} for a, b, c in TRIPLE_RE.findall(text)]}

for name, d in DATA.items():
    reqs = {f"p{i}": LLM.body([{"role": "system", "content": OPENIE_SYS}, {"role": "user", "content": p}],
                              max_tokens=4000, schema=OpenIE)
            for i, p in enumerate(d["passages"])}
    res = llm.run(f"{name}_openie", reqs, parse=OpenIE)
    d["openie"] = [res[f"p{i}"]["parsed"] or salvage(res[f"p{i}"].get("text", "")) for i in range(len(d["passages"]))]
    bad = sum(res[f"p{i}"]["parsed"] is None for i in range(len(d["passages"])))
    nt = sum(len(x["triples"]) for x in d["openie"])
    print(f"{name}: {len(reqs):,} extraction requests, {bad} cut off (complete triples kept), {nt:,} triples "
          f"({nt / len(reqs):.1f} per passage)")

musique: 11,656 extraction requests, 1 cut off (complete triples kept), 126,546 triples (10.9 per passage)


2wikimultihopqa: 6,119 extraction requests, 1 cut off (complete triples kept), 63,801 triples (10.4 per passage)


In [5]:
norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(s).lower())).strip()

def build_graph(d):
    # Nodes: passages 0..P-1, then entities. Edges: entity-entity (triple), entity-passage (mention), synonyms.
    n_p = len(d["passages"])
    ent_id, ent_name = {}, []
    def eid(s):
        k = norm(s)
        if not k:
            return None
        if k not in ent_id:
            ent_id[k] = n_p + len(ent_name); ent_name.append(s)
        return ent_id[k]
    rows, cols, facts = [], [], []
    for pi, ex in enumerate(d["openie"]):
        for e in ex["entities"]:
            j = eid(e)
            if j is not None:
                rows += [pi, j]; cols += [j, pi]
        for t in ex["triples"]:
            a, b = eid(t["subject"]), eid(t["object"])
            if a is None or b is None or a == b:
                continue
            rows += [a, b, pi, a, pi, b]; cols += [b, a, a, pi, b, pi]
            facts.append((a, b, pi, f"{t['subject']} {t['predicate']} {t['object']}"))
    d["n_p"], d["ent_name"], d["facts"] = n_p, ent_name, facts
    # Synonym edges between near-duplicate entity names (HippoRAG 2), computed on entity-name embeddings.
    E = llm.embed(f"{d['name']}_entities", ent_name, cap=200)
    syn = 0
    for s in range(0, len(ent_name), 512):
        S = E[s:s + 512] @ E.T
        ii, jj = np.where(S >= 0.9)
        for a, b in zip(ii + s, jj):
            if a < b:
                rows += [n_p + a, n_p + b]; cols += [n_p + b, n_p + a]; syn += 1
    n = n_p + len(ent_name)
    A = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n, n))
    A.data[:] = 1.0                                  # duplicate edges count once
    deg = np.asarray(A.sum(1)).ravel(); deg[deg == 0] = 1
    d["T"] = (sp.diags(1 / deg) @ A).T.tocsr()        # column-stochastic transition matrix
    d["F"] = llm.embed(f"{d['name']}_facts", [f[3] for f in facts], cap=300)
    return n, syn

for name, d in DATA.items():
    d["name"] = name
    n, syn = build_graph(d)
    print(f"{name}: graph with {d['n_p']:,} passage + {len(d['ent_name']):,} entity nodes, "
          f"{len(d['facts']):,} triples, {syn:,} synonym edges")

musique: graph with 11,656 passage + 148,016 entity nodes, 126,335 triples, 14,424 synonym edges


2wikimultihopqa: graph with 6,119 passage + 68,410 entity nodes, 63,711 triples, 10,448 synonym edges


## Two retrievers with the same budget of 5 passages

**Vector:** the 5 passages most similar to the question. **Graph:** the question embedding is compared with every
triple; the entities of the 5 best triples (weighted by similarity) plus 0.05 × the dense score of the 20 best
passages form the personalisation vector of a personalised PageRank (damping 0.5); the 5 passages with the highest
PageRank go to the answerer. Recall@5 counts how many gold supporting paragraphs each retriever found.

In [6]:
K = 5

def ppr(T, pers, alpha=0.5, iters=40):
    p = pers / pers.sum(); x = p.copy()
    for _ in range(iters):
        x = (1 - alpha) * p + alpha * (T @ x)
    return x

def retrieve(d, qi):
    q = d["Q"][qi]
    dense = d["P"] @ q
    vec = list(np.argsort(-dense)[:K])
    pers = np.zeros(d["T"].shape[0], np.float64)
    fs = d["F"] @ q if len(d["facts"]) else np.zeros(0)
    for j in np.argsort(-fs)[:5]:
        a, b, pi, _ = d["facts"][j]
        pers[a] += fs[j]; pers[b] += fs[j]
    for j in np.argsort(-dense)[:20]:
        pers[j] += 0.05 * max(dense[j], 0)
    if pers.sum() == 0:
        return vec, vec
    x = ppr(d["T"], pers)[:d["n_p"]]
    return vec, list(np.argsort(-x)[:K])

def recall(d, q, ids):
    if not q["support"]:
        return float("nan")
    got = {d["passages"][i] for i in ids}
    return len(q["support"] & got) / len(q["support"])

t0 = time.time()
for name, d in DATA.items():
    d["ret"] = [retrieve(d, i) for i in range(len(d["questions"]))]
    rv = np.nanmean([recall(d, q, v) for q, (v, g) in zip(d["questions"], d["ret"])])
    rg = np.nanmean([recall(d, q, g) for q, (v, g) in zip(d["questions"], d["ret"])])
    print(f"{name}: recall@5 of gold supporting paragraphs  vector {rv:.1%}  |  graph {rg:.1%}")
print(f"retrieval for {sum(len(d['questions']) for d in DATA.values()):,} questions in {time.time() - t0:.0f} s")

musique: recall@5 of gold supporting paragraphs  vector 55.3%  |  graph 58.4%


2wikimultihopqa: recall@5 of gold supporting paragraphs  vector 70.6%  |  graph 84.4%
retrieval for 2,000 questions in 68 s


## Answers (one Batch job) and EM / F1 scoring

In [7]:
ANSWER_SYS = ("Answer the question using the passages. Reason step by step in at most three short sentences, then "
              "give the final answer on the last line as 'Answer: <answer>'. The answer must be a short phrase "
              "(an entity, date, number or yes/no), not a sentence.")

def ctx(d, ids):
    return "\n\n".join(f"[{n + 1}] {d['passages'][i]}" for n, i in enumerate(ids))

def final(text):
    m = re.findall(r"Answer:\s*(.+)", text)
    return (m[-1] if m else text.strip().splitlines()[-1] if text.strip() else "").strip()

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g); same = sum(common.values())
    if same == 0:
        return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

for name, d in DATA.items():
    reqs = {}
    for qi, q in enumerate(d["questions"]):
        for m, ids in zip(("vector", "graph"), d["ret"][qi]):
            reqs[f"{m}|{q['qid']}"] = LLM.body([{"role": "system", "content": ANSWER_SYS},
                                               {"role": "user", "content": f"{ctx(d, ids)}\n\nQuestion: {q['question']}"}],
                                              max_tokens=300)
    res = llm.run(f"{name}_answers", reqs)
    d["scores"] = []
    for qi, q in enumerate(d["questions"]):
        row = {"group": q["group"]}
        for m, ids in zip(("vector", "graph"), d["ret"][qi]):
            pred = final(res[f"{m}|{q['qid']}"]["text"])
            row[m] = {"pred": pred, "em": float(max(normalize(pred) == normalize(g) for g in q["golds"])),
                      "f1": max(f1(pred, g) for g in q["golds"]), "recall": recall(d, q, ids)}
        d["scores"].append(row)
    print(f"{name}: {len(reqs):,} answers")

musique: 2,000 answers


2wikimultihopqa: 2,000 answers


## Results with 95% confidence intervals

EM intervals are Wilson intervals; the *graph − vector* column is a paired bootstrap over questions on F1. An
interval that excludes 0 is a difference this sample can resolve.

In [8]:
SUMMARY = {}
def table(name, d):
    print(f"\n{name}  (n = {len(d['scores'])})")
    print(f"{'group':<20}{'n':>5}{'EM vector':>18}{'EM graph':>18}{'F1 vector':>11}{'F1 graph':>10}{'F1 graph-vector [95% CI]':>28}"
          f"{'recall@5 v/g':>15}")
    groups = sorted({s["group"] for s in d["scores"]}) + ["ALL"]
    for g in groups:
        rows = [s for s in d["scores"] if g == "ALL" or s["group"] == g]
        n = len(rows)
        out = {}
        for m in ("vector", "graph"):
            k = int(sum(r[m]["em"] for r in rows)); lo, hi = wilson(k, n)
            out[m] = (k / n, lo, hi, np.mean([r[m]["f1"] for r in rows]), np.nanmean([r[m]["recall"] for r in rows]))
        dm, dlo, dhi = paired_boot([r["vector"]["f1"] for r in rows], [r["graph"]["f1"] for r in rows])
        SUMMARY[(name, g)] = (n, out, (dm, dlo, dhi))
        em = lambda t: f"{t[0]:.1%} [{t[1]:.0%}-{t[2]:.0%}]"
        print(f"{g:<20}{n:>5}{em(out['vector']):>18}{em(out['graph']):>18}{out['vector'][3]:>11.1%}{out['graph'][3]:>10.1%}"
              f"{f'{dm:+.1%} [{dlo:+.1%}, {dhi:+.1%}]':>28}{f'{out['vector'][4]:.0%}/{out['graph'][4]:.0%}':>15}")

for name, d in DATA.items():
    table(name, d)


musique  (n = 1000)
group                   n         EM vector          EM graph  F1 vector  F1 graph    F1 graph-vector [95% CI]   recall@5 v/g
2-hop                 518   29.7% [26%-34%]   38.0% [34%-42%]      43.7%     51.6%       +7.9% [+4.2%, +11.6%]        66%/72%
3-hop                 316   25.9% [21%-31%]   26.9% [22%-32%]      36.6%     39.4%        +2.9% [-2.1%, +7.9%]        50%/50%
4-hop                 166   15.7% [11%-22%]    12.7% [8%-19%]      24.3%     22.6%        -1.7% [-6.3%, +2.9%]        33%/32%
ALL                  1000   26.2% [24%-29%]   30.3% [28%-33%]      38.2%     43.0%        +4.7% [+2.1%, +7.4%]        55%/58%

2wikimultihopqa  (n = 1000)
group                   n         EM vector          EM graph  F1 vector  F1 graph    F1 graph-vector [95% CI]   recall@5 v/g
bridge_comparison     235   58.7% [52%-65%]   63.8% [58%-70%]      61.9%     67.6%       +5.7% [-2.2%, +13.3%]        50%/71%
comparison            244   86.9% [82%-91%]   80.3% [75%-85%]      8

### One question, two retrievals

A question the graph answered and the vector baseline did not (if there is one), with the passages each retriever
returned.

In [9]:
d = DATA["musique"]
idx = next((i for i, s in enumerate(d["scores"]) if s["graph"]["em"] and not s["vector"]["em"]), 0)
q, (v_ids, g_ids) = d["questions"][idx], d["ret"][idx]
print(f"Q: {q['question']}\nGold: {q['golds'][0]}   (supporting paragraphs: {len(q['support'])})")
for m, ids in (("vector", v_ids), ("graph", g_ids)):
    print(f"\n{m.upper()} -> predicted {d['scores'][idx][m]['pred']!r}, recall@5 {d['scores'][idx][m]['recall']:.0%}")
    for i in ids:
        mark = "*" if d["passages"][i] in q["support"] else " "
        print(f"  {mark} {d['passages'][i][:110].replace(chr(10), ' | ')}")
print("\n(* = gold supporting paragraph)")

Q: How many episodes are in season 5 of the series with The Bag or the Bat?
Gold: 12   (supporting paragraphs: 2)

VECTOR -> predicted '16 episodes', recall@5 50%
  * The Bag or the Bat | "The Bag or the Bat" is the pilot episode of the Showtime original series "Ray Donovan", an
    Arrested Development (season 5) | The fifth season of the television comedy series Arrested Development premiere
    Samurai Jack (season 5) | The fifth season of Samurai Jack is the final season of the animated series. This seas
    List of Orange Is the New Black episodes | As of June 9, 2017, 65 episodes of Orange Is the New Black have been 
    The Sopranos (season 6) | The sixth and final season of the HBO drama series "The Sopranos" was broadcast in two

GRAPH -> predicted '12', recall@5 50%
    Arrested Development (season 5) | The fifth season of the television comedy series Arrested Development premiere
    Glee (season 5) | The fifth season of the Fox musical comedy - drama television series Glee 

## What the graph cost (Slides 34, 37)

In [10]:
LEDGER = llm.print_ledger()
for name, d in DATA.items():
    ie = LEDGER.get(f"{name}_openie", {})
    an = LEDGER.get(f"{name}_answers", {})
    print(f"{name}: graph index = {ie.get('calls', 0):,} LLM calls ({(ie.get('in', 0) + ie.get('out', 0)) / 1e6:.1f}M tokens, "
          f"${ie.get('usd', 0):.2f}); answering both systems = ${an.get('usd', 0):.2f}")

stage                           calls     input tok   output tok      USD  via Batch
2wikimultihopqa_answers         2,000     1,704,967       44,864     0.10       100%
2wikimultihopqa_openie          6,119     2,024,810    2,585,174     0.75       100%
musique_answers                 2,000     1,581,240       40,483     0.09       100%
musique_openie                 11,656     3,914,741    5,072,714     1.46       100%
(embeddings)                        0     5,924,546            0     0.12         0%
TOTAL                          21,775    15,150,304    7,743,235     2.52
musique: graph index = 11,656 LLM calls (9.0M tokens, $1.46); answering both systems = $0.09
2wikimultihopqa: graph index = 6,119 LLM calls (4.6M tokens, $0.75); answering both systems = $0.10


In [11]:
print("=" * 92)
print(f"RESULTS SUMMARY — Notebook 11: multi-hop QA over text, {MODE} run, gpt-6-luna (no reasoning), top-5 passages")
print("=" * 92)
print(f"{'benchmark':<18}{'n':>6}{'EM vector':>12}{'EM graph':>11}{'F1 graph - vector [95% CI]':>30}{'recall@5 vector / graph':>26}")
verdicts = {}
for name in DATA:
    n, out, (dm, dlo, dhi) = SUMMARY[(name, "ALL")]
    verdicts[name] = "graph better" if dlo > 0 else "vector better" if dhi < 0 else "no resolvable difference"
    print(f"{name:<18}{n:>6}{out['vector'][0]:>12.1%}{out['graph'][0]:>11.1%}"
          f"{f'{dm:+.1%} [{dlo:+.1%}, {dhi:+.1%}]':>30}{f'{out['vector'][4]:.1%} / {out['graph'][4]:.1%}':>26}")
print("\nBy hop count (MuSiQue, F1 graph - vector):",
      "; ".join(f"{g} {SUMMARY[('musique', g)][2][0]:+.1%} [{SUMMARY[('musique', g)][2][1]:+.1%}, {SUMMARY[('musique', g)][2][2]:+.1%}]"
                for g in sorted({s['group'] for s in DATA['musique']['scores']})))
print("Verdict:", ", ".join(f"{k}: {v}" for k, v in verdicts.items()))
print("\nLecture message (Slides 32-41): compare with Notebook 8. When the graph has to be extracted from text, its"
      "\nadvantage is whatever the verdict line says — not the 40-60 points MetaQA showed with a hand-made graph."
      "\nMeasure the graph you can actually build against a strong vector baseline before paying for it.")

RESULTS SUMMARY — Notebook 11: multi-hop QA over text, replay run, gpt-6-luna (no reasoning), top-5 passages
benchmark              n   EM vector   EM graph    F1 graph - vector [95% CI]   recall@5 vector / graph
musique             1000       26.2%      30.3%          +4.7% [+2.1%, +7.4%]             55.3% / 58.4%
2wikimultihopqa     1000       45.0%      53.3%        +11.7% [+8.6%, +14.8%]             70.6% / 84.4%

By hop count (MuSiQue, F1 graph - vector): 2-hop +7.9% [+4.2%, +11.6%]; 3-hop +2.9% [-2.1%, +7.9%]; 4-hop -1.7% [-6.3%, +2.9%]
Verdict: musique: graph better, 2wikimultihopqa: graph better

Lecture message (Slides 32-41): compare with Notebook 8. When the graph has to be extracted from text, its
advantage is whatever the verdict line says — not the 40-60 points MetaQA showed with a hand-made graph.
Measure the graph you can actually build against a strong vector baseline before paying for it.
